# NumPy Fundamentals: Array Creation, Matrices, Random Numbers & Vectorization

My learning notes on NumPy, covering:
1. Creating sequences: `arange`, `linspace`, `logspace`
2. Special arrays: `zeros`, `ones`, `eye`
3. Diagonals and triangular matrices: `diag`, `tril`, `triu`
4. Random number generation
5. Axis-based aggregation
6. Vectorization and broadcasting

In [1]:
import numpy as np

## 1. Creating Sequences

### `range` vs `np.arange`
Python's `range(start, stop, step)` only supports integers and returns a lazy range object. `np.arange` has the same signature but supports floats and a `dtype`, and returns an array. The `stop` value is **excluded**.

In [2]:
list(range(0, 10, 2))

[0, 2, 4, 6, 8]

In [3]:
np.arange(1, 16, 4, dtype=np.float32)

array([ 1.,  5.,  9., 13.], dtype=float32)

In [4]:
np.arange(1, 16, 4.5)   # float step works

array([ 1. ,  5.5, 10. , 14.5])

### `np.linspace(start, stop, num)`
Gives `num` evenly spaced points, with **both endpoints included**. You choose the *count* of points, and NumPy works out the step size.

In [5]:
np.linspace(1, 5, 21)

array([1. , 1.2, 1.4, 1.6, 1.8, 2. , 2.2, 2.4, 2.6, 2.8, 3. , 3.2, 3.4,
       3.6, 3.8, 4. , 4.2, 4.4, 4.6, 4.8, 5. ])

Passing an integer `dtype` truncates the values (decimals are dropped, not rounded), so be careful:

In [6]:
np.linspace(1, 5, 10, dtype=np.int32)

array([1, 1, 1, 2, 2, 3, 3, 4, 4, 5], dtype=int32)

In [7]:
np.linspace(1, 5, 10, dtype=np.float32)

array([1.       , 1.4444444, 1.8888888, 2.3333333, 2.7777777, 3.2222223,
       3.6666667, 4.111111 , 4.5555553, 5.       ], dtype=float32)

### `np.logspace(start, stop, num, base=10)`
Points evenly spaced on a **log scale**: the values are `base ** linspace(start, stop, num)`.

In [8]:
np.logspace(1, 2, 5)   # 10^1 to 10^2

array([ 10.        ,  17.7827941 ,  31.6227766 ,  56.23413252,
       100.        ])

In [9]:
np.logspace(0, 10, num=11, base=2, dtype=np.int32)   # powers of 2: 2^0 ... 2^10

array([   1,    2,    4,    8,   16,   32,   64,  128,  256,  512, 1024],
      dtype=int32)

## 2. Special Arrays: `zeros`, `ones`, `eye`

In [10]:
np.zeros(10)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [11]:
np.ones((3, 3), dtype=np.int32)

array([[1, 1, 1],
       [1, 1, 1],
       [1, 1, 1]], dtype=int32)

In [12]:
np.ones((3, 3), dtype=np.int32) * 7   # fill with any constant by scaling ones

array([[7, 7, 7],
       [7, 7, 7],
       [7, 7, 7]], dtype=int32)

In [13]:
np.eye(4)   # identity matrix

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

## 3. Diagonals and Triangular Matrices

`np.diag` behaves differently depending on its input:
- **2D input** gives the diagonal as a 1D array
- **1D input** gives a 2D matrix with that array on the diagonal

In [14]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=np.int32)

diagonal_elements = np.diag(A)
print(diagonal_elements)

[1 5 9]


In [15]:
# np.diag applied twice: extract the diagonal, then rebuild a diagonal matrix from it
diagonal_matrix = np.diag(np.diag(A))
print(diagonal_matrix)

[[1 0 0]
 [0 5 0]
 [0 0 9]]


In [16]:
lower_tri = np.tril(A)   # keeps lower triangle, zeros above
print(lower_tri)

[[1 0 0]
 [4 5 0]
 [7 8 9]]


In [17]:
upper_tri = np.triu(A)   # keeps upper triangle, zeros below
print(upper_tri)

[[1 2 3]
 [0 5 6]
 [0 0 9]]


In [18]:
A.shape

(3, 3)

### The `k` parameter (diagonal offset)
- `k = 0`: main diagonal (default)
- `k > 0`: diagonals above the main one
- `k < 0`: diagonals below the main one

In [19]:
x = np.arange(9).reshape((3, 3))
print(x)

[[0 1 2]
 [3 4 5]
 [6 7 8]]


In [20]:
print(np.diag(x))      # main diagonal

[0 4 8]


In [21]:
print(np.diag(x, 1))   # one above

[1 5]


In [22]:
print(np.diag(x, -1))  # one below

[3 7]


In [23]:
np.tril(x)

array([[0, 0, 0],
       [3, 4, 0],
       [6, 7, 8]])

In [24]:
np.tril(x, -1)   # excludes the main diagonal

array([[0, 0, 0],
       [3, 0, 0],
       [6, 7, 0]])

In [25]:
np.triu(x)

array([[0, 1, 2],
       [0, 4, 5],
       [0, 0, 8]])

In [26]:
np.triu(x, 1)    # excludes the main diagonal

array([[0, 1, 2],
       [0, 0, 5],
       [0, 0, 0]])

## 4. Random Number Generation

| Function | Output |
|---|---|
| `np.random.rand(d0, d1)` | Uniform floats in [0, 1) |
| `np.random.randn(d0, d1)` | Standard normal (mean 0, std 1) |
| `np.random.randint(low, high, size)` | Random integers in [low, high) |

Setting a seed makes results reproducible, so the outputs below are identical on every run.

In [27]:
np.random.seed(42)

In [28]:
print(np.random.rand(2, 3))

[[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]]


In [29]:
print(np.random.randn(2, 3))

[[ 1.57921282  0.76743473 -0.46947439]
 [ 0.54256004 -0.46341769 -0.46572975]]


In [30]:
print(np.random.randint(1, 100, 10))

[76 58 22 89 49 91 59 42 92 60]


In [31]:
print(np.random.randint(1, 100, (4, 4)))

[[80 15 62 62]
 [47 62 51 55]
 [64  3 51  7]
 [21 73 39 18]]


In [32]:
print(np.random.randint(1, 3, 10))   # coin toss: 1 = heads, 2 = tails

[2 1 2 2 2 2 1 2 1 2]


## 5. Aggregation Along an Axis

- `axis=0` collapses **rows**, giving one result per **column**
- `axis=1` collapses **columns**, giving one result per **row**

In [33]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]])
np.sum(arr, axis=0)   # column sums

array([5, 7, 9])

In [34]:
np.sum(arr, axis=1)   # row sums

array([ 6, 15])

## 6. Vectorization and Broadcasting

NumPy applies operations to whole arrays at once, without Python loops. This is faster and cleaner.

**Broadcasting** lets arrays of different shapes work together: a scalar is stretched to match the array, and a 1D array is stretched across the rows of a 2D array.

In [35]:
arr = np.array([10, 20, 30])
print(arr + 5)   # scalar broadcast

[15 25 35]


In [36]:
arr01 = np.array([[4, 5, 6],
                  [7, 8, 9]])
print(arr01 + arr)   # (2,3) + (3,) -> arr is added to each row

[[14 25 36]
 [17 28 39]]


### Vectorized vs loop-based element-wise addition

In [37]:
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])
A + B

array([5, 7, 9])

In [38]:
[int(x + y) for x, y in zip(A, B)]   # same result, but needs a Python loop

[5, 7, 9]

## Key Takeaways
- `arange` is step-based, `linspace` is count-based, `logspace` is count-based on a log scale
- `diag`, `tril`, `triu` all accept an offset `k`
- `axis=0` works down the columns, `axis=1` works across the rows
- Prefer vectorized operations over Python loops